# Diamond & PlaNet World Model Tutorial

This notebook demonstrates how to train and perform inference with the Diamond and PlaNet world models from TorchWM.

## What is Diamond?

**DIAMOND** (DIffusion As a Model Of eNvironment Dreams) is a model-based RL agent that uses diffusion models for world modeling:

- **Diffusion World Model**: Learns to generate future observations using diffusion
- **Reward/Termination Model**: Predicts rewards and episode termination
- **Actor-Critic**: Learns policies via imagined rollouts in the diffusion model
- Works on Atari games (Atari 100k benchmark)

## What is PlaNet?

**PlaNet** (Planning Network) uses a recurrent state-space model (RSSM) for latent planning:

- **RSSM**: Latent dynamics model with deterministic and stochastic components
- **VAE**: For encoding images to latent space
- **Planning**: Uses learned latent dynamics for model-based planning
- Works well on continuous control tasks (DMC)

In [ ]:
# Setup and Imports
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# TorchWM imports
from world_models.configs.diamond_config import DiamondConfig
from world_models.inference.operators import PlaNetOperator
from world_models.training.train_diamond import DiamondAgent

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Diamond World Model

Diamond trains entirely within a diffusion world model. It uses EDM (Elucidating the Design Space of Diffusion Models) sampling.

In [ ]:
# Diamond Configuration
diamond_config = DiamondConfig()

# Customize for quick demo
diamond_config.game = "Breakout-v5"
diamond_config.obs_size = 64
diamond_config.num_epochs = 2  # Short for demo
diamond_config.environment_steps_per_epoch = 200
diamond_config.training_steps_per_epoch = 20
diamond_config.batch_size = 16
diamond_config.device = "cuda" if torch.cuda.is_available() else "cpu"
diamond_config.log_interval = 1
diamond_config.eval_interval = 1

print("=== Diamond Configuration ===")
print(f"Game: {diamond_config.game}")
print(f"Observation size: {diamond_config.obs_size}")
print(f"Num epochs: {diamond_config.num_epochs}")
print(f"Environment steps per epoch: {diamond_config.environment_steps_per_epoch}")
print(f"Imagination horizon: {diamond_config.imagination_horizon}")

## Training Diamond

Diamond training involves:
1. **Experience Collection**: Gather data from the environment
2. **Diffusion Model Update**: Train the world model to predict next observations
3. **Reward Model Update**: Train reward and termination predictors
4. **Actor-Critic Update**: Train policy via imagined rollouts in the diffusion model

In [ ]:
# Create and train Diamond agent
print("Creating Diamond agent...")
diamond_agent = DiamondAgent(diamond_config)

print("\n=== Diamond Components ===")
print(f"Diffusion model: {type(diamond_agent.diffusion_model).__name__}")
print(f"Reward model: {type(diamond_agent.reward_model).__name__}")
print(f"Actor-Critic: {type(diamond_agent.actor_critic).__name__}")

print("\nStarting training...")
print("=" * 50)
diamond_agent.train()
print("=" * 50)
print("Training complete!")

In [ ]:
# Evaluate Diamond agent
print("\nEvaluating Diamond agent...")
eval_returns = []

for episode in range(3):
    eval_return = diamond_agent.evaluate(num_episodes=1)
    eval_returns.append(eval_return)
    print(f"Episode {episode + 1}: Return = {eval_return:.2f}")

print(f"\nMean evaluation return: {np.mean(eval_returns):.2f} +/- {np.std(eval_returns):.2f}")

## PlaNet / RSSM Model

PlaNet uses an RSSM (Recurrent State Space Model) for learning latent dynamics. Let's also demonstrate training PlaNet.

In [ ]:
# PlaNet/RSSM Configuration and Training
from world_models.models.rssm import RecurrentStateSpaceModel
from world_models.vision.planet_encoder import ConvEncoder
from world_models.vision.planet_decoder import ConvDecoder
from world_models.memory.planet_memory import Memory
from world_models.controller.rssm_policy import RSSMPolicy
import torch.optim as optim

# RSSM configuration
state_dim = 64
hidden_dim = 200
embedding_dim = 1024
action_dim = 6  # For DMC cartpole

# Create models
rssm = RecurrentStateSpaceModel(
    state_dim=state_dim,
    hidden_dim=hidden_dim,
    embedding_dim=embedding_dim,
    action_dim=action_dim,
    num_layers=1,
)

encoder = ConvEncoder(embedding_dim=embedding_dim)
decoder = ConvDecoder(embedding_dim=embedding_dim, state_dim=state_dim)
policy = RSSMPolicy(state_dim=state_dim, hidden_dim=hidden_dim, action_dim=action_dim)

# Move to device
rssm = rssm.to(device)
encoder = encoder.to(device)
decoder = decoder.to(device)
policy = policy.to(device)

print("=== PlaNet/RSSM Components ===")
print(f"RSSM: {type(rssm).__name__}")
print(f"Encoder: {type(encoder).__name__}")
print(f"Decoder: {type(decoder).__name__}")
print(f"Policy: {type(policy).__name__}")

In [ ]:
# Create memory buffer and optimizer
memory = Memory(
    capacity=10000,
    obs_shape=(3, 64, 64),
    action_dim=action_dim,
)

optimizer = optim.Adam(
    list(rssm.parameters()) + list(encoder.parameters()) + list(decoder.parameters()),
    lr=1e-3
)

print(f"Memory capacity: {memory.capacity}")
print(f"Observation shape: {memory.obs_shape}")

In [ ]:
# Collect some experience from DMC environment
from world_models.envs.dmc import DeepMindControlEnv
import world_models.envs.wrappers as env_wrapper

env = DeepMindControlEnv('cartpole-balance', seed=42, size=(64, 64))
env = env_wrapper.ActionRepeat(env, 2)
env = env_wrapper.NormalizeActions(env)

print("Collecting experience from DMC environment...")

# Collect random episodes to fill memory
for episode in range(5):
    obs, _ = env.reset()
    obs = obs.to(torch.float32) / 255.0
    done = False
    steps = 0
    max_steps = 200
    
    while not done and steps < max_steps:
        action = env.action_space.sample()
        next_obs, reward, done, _ = env.step(action)
        next_obs = next_obs.to(torch.float32) / 255.0
        
        # Store in memory
        memory.add(obs, action, reward, done)
        
        obs = next_obs
        steps += 1
    
    print(f"Episode {episode + 1}: {steps} steps")

print(f"\nMemory size: {len(memory)}")

In [ ]:
# Training loop for PlaNet/RSSM
from torch.distributions import Normal
from torch.distributions.kl import kl_divergence
import torch.nn.functional as F

print("\nTraining PlaNet/RSSM...")
print("=" * 50)

losses = []
num_training_steps = 50
batch_size = 32
horizon = 20

for step in range(num_training_steps):
    # Sample batch from memory
    (obs_batch, action_batch, reward_batch, done_batch), lengths = memory.sample(
        batch_size, horizon, time_first=True
    )
    
    # Convert to tensors
    obs_batch = torch.tensor(obs_batch).float().to(device)
    action_batch = torch.tensor(action_batch).float().to(device)
    reward_batch = torch.tensor(reward_batch).float().to(device)
    
    # Encode observations
    embeddings = encoder(obs_batch)
    
    # Get initial states
    h_t, s_t = rssm.get_init_state(embeddings[0])
    
    # Forward pass through RSSM
    states, priors, posteriors = [], [], []
    
    for i in range(horizon - 1):
        h_t = rssm.deterministic_state_fwd(h_t, s_t, action_batch[i])
        prior = rssm.state_prior(h_t)
        posterior = rssm.state_posterior(h_t, embeddings[i + 1])
        
        states.append(h_t)
        priors.append(prior)
        posteriors.append(posterior)
        
        # Sample from posterior
        s_t = Normal(*posterior).rsample()
    
    # Stack sequences
    states = torch.stack(states)
    posterior_samples = torch.stack([Normal(*p).rsample() for p in posteriors])
    
    # Reconstruction loss
    recon = decoder(states, posterior_samples)
    rec_loss = F.mse_loss(recon, obs_batch[1:], reduction='sum') / batch_size
    
    # KL loss
    prior_dist = Normal(*map(torch.stack, zip(*priors)))
    posterior_dist = Normal(*map(torch.stack, zip(*posteriors)))
    kld_loss = kl_divergence(posterior_dist, prior_dist).sum(-1).mean()
    
    # Reward loss
    reward_pred = rssm.pred_reward(states, posterior_samples)
    rew_loss = F.mse_loss(reward_pred, reward_batch[:-1])
    
    # Total loss
    loss = rec_loss + kld_loss + rew_loss
    
    # Update
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    
    if step % 10 == 0:
        print(f"Step {step}: Loss = {loss.item():.4f}, Rec = {rec_loss.item():.4f}, KL = {kld_loss.item():.4f}")

print("=" * 50)
print("PlaNet training complete!")

In [ ]:
# Plot training losses
plt.figure(figsize=(10, 4))
plt.plot(losses)
plt.xlabel('Training Step')
plt.ylabel('Loss')
plt.title('PlaNet/RSSM Training Loss')
plt.grid(True)
plt.savefig('planet_training_loss.png', dpi=150)
plt.show()

print("Training loss plot saved to planet_training_loss.png")

## PlaNet Inference Operator

Use `PlaNetOperator` for preprocessing during inference.

In [ ]:
# Using PlaNetOperator
from PIL import Image

planet_operator = PlaNetOperator(state_dim=64, action_dim=6)

# Example: process state and action
dummy_obs = Image.new('RGB', (64, 64), color='gray')
dummy_action = [0.0, 0.5, 0.0, 0.0, 0.0, 0.0]
dummy_reward = 1.0
dummy_done = False

processed = planet_operator.process({
    'obs': dummy_obs,
    'action': dummy_action,
    'reward': dummy_reward,
    'done': dummy_done
})

print("=== PlaNet Operator Demo ===")
print("Processed observation shape:", processed['obs'].shape)
print("Processed action shape:", processed['action'].shape)
print("Processed reward shape:", processed['reward'].shape)
print("Processed done shape:", processed['done'].shape)

## Saving and Loading Models

In [ ]:
# Save Diamond checkpoint
diamond_save_path = "checkpoints/diamond/notebook_model.pt"
diamond_agent.save_checkpoint("notebook_diamond.pt")
print(f"Diamond model saved to notebook_diamond.pt")

# Save PlaNet/RSSM checkpoint
planet_save_path = "checkpoints/planet/notebook_rssm.pt"
os.makedirs("checkpoints/planet", exist_ok=True)
torch.save({
    'rssm': rssm.state_dict(),
    'encoder': encoder.state_dict(),
    'decoder': decoder.state_dict(),
    'optimizer': optimizer.state_dict(),
}, planet_save_path)
print(f"PlaNet model saved to {planet_save_path}")

## Comparison: Diamond vs PlaNet

In [ ]:
print("=== Diamond vs PlaNet Comparison ===")

comparison = """
| Feature | Diamond | PlaNet |
|---------|---------|--------|
| World Model | Diffusion (EDM) | RSSM (VAE-based) |
| Observation Generation | Diffusion sampling | Deterministic decoder |
| Domain | Atari (discrete) | DMC (continuous) |
| Planning | Imagined rollouts | Latent planning |
| Sample Efficiency | High (100k) | High |
| Training Complexity | Medium | Medium |
"""
print(comparison)

print("\n=== When to Use Which? ===")
print("""
Use Diamond when:
  - Working with Atari games
  - Need high-quality image generation
  - Working with discrete action spaces

Use PlaNet when:
  - Working with continuous control (DMC)
  - Need fast inference
  - Want to use latent planning
""")

## Summary

In this tutorial, you learned:

1. **Diamond**: Diffusion-based world model for Atari
   - Diffusion UNet for observation prediction
   - Reward/termination models
   - Actor-critic trained via imagined rollouts
   - Trained and evaluated on Breakout

2. **PlaNet/RSSM**: Latent dynamics model for continuous control
   - Recurrent State Space Model
   - Variational training objective
   - Trained on DMC cartpole

3. **Inference Operators**: How to use PlaNetOperator for preprocessing

4. **Saving/Loading**: How to save model checkpoints